# Neural Network Experiments — Supervised Fraud Detection

All experiments are fully **supervised** classification. Key techniques:
- **Focal loss** — down-weights easy negatives, focuses on hard fraud cases
- **Heavy class weighting** — multiple weight scales tested
- **SMOTE oversampling** — at aggressive ratios (0.3, 0.5, 1.0)
- **Architecture search** — shallow vs deep, with dropout/batch norm
- **Ensemble** — combine best NN + RF + XGBoost

Each experiment reports per-class recall at default, F1-optimal, and F2-optimal thresholds.

In [9]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, precision_recall_curve,
    average_precision_score, recall_score, precision_score, f1_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow {tf.__version__}')
print('All imports OK')

TensorFlow 2.20.0
All imports OK


## 1. Data Loading & Preprocessing

In [10]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'mlg-ulb/creditcardfraud',
    'creditcard.csv'
)

print(f"Dataset shape: {df.shape}")
print(f"Fraud ratio: {df['Class'].mean():.4%}")

new_df = df.copy()
new_df['Amount'] = RobustScaler().fit_transform(new_df['Amount'].to_numpy().reshape(-1, 1))
new_df['Time']   = StandardScaler().fit_transform(new_df[['Time']])
new_df = new_df.sample(frac=1, random_state=42)

train, temp = train_test_split(new_df, test_size=0.2, stratify=new_df['Class'], random_state=42)
test,  val  = train_test_split(temp,   test_size=0.5, stratify=temp['Class'],   random_state=42)

x_train = train.drop(columns=['Class']); y_train = train['Class']
x_test  = test.drop(columns=['Class']);  y_test  = test['Class']
x_val   = val.drop(columns=['Class']);   y_val   = val['Class']

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")
print(f"Val fraud count: {y_val.sum()} / {len(y_val)}")

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
print(f"Neg/Pos ratio: {neg_count/pos_count:.1f}:1")

Dataset shape: (284807, 31)
Fraud ratio: 0.1727%
Train: (227845, 30), Val: (28481, 30), Test: (28481, 30)
Val fraud count: 49 / 28481
Neg/Pos ratio: 577.3:1


## 2. Shared Evaluation Helper

In [11]:
results = []

def evaluate(name, y_true, y_proba):
    """Evaluate at default, F1-optimal, and F2-optimal thresholds with per-class recall."""
    prec_arr, rec_arr, thr_arr = precision_recall_curve(y_true, y_proba)
    n = len(thr_arr)
    eps = 1e-9

    f1 = np.array([2*prec_arr[j+1]*rec_arr[j+1] / (prec_arr[j+1]+rec_arr[j+1]+eps) for j in range(n)])
    f2 = np.array([5*prec_arr[j+1]*rec_arr[j+1] / (4*prec_arr[j+1]+rec_arr[j+1]+eps) for j in range(n)])

    pr_auc = average_precision_score(y_true, y_proba)

    pred_def = (y_proba >= 0.5).astype(int)
    j1 = np.argmax(f1)
    pred_f1 = (y_proba >= thr_arr[j1]).astype(int)
    j2 = np.argmax(f2)
    pred_f2 = (y_proba >= thr_arr[j2]).astype(int)

    for label, pred in [
        ('default (0.5)', pred_def),
        (f'F1-opt ({thr_arr[j1]:.4f})', pred_f1),
        (f'F2-opt ({thr_arr[j2]:.4f})', pred_f2),
    ]:
        results.append({
            'Experiment': name, 'Threshold': label,
            'Precision (fraud)': precision_score(y_true, pred),
            'Recall (fraud)': recall_score(y_true, pred),
            'Recall (legit)': recall_score(y_true, pred, pos_label=0),
            'F1 (fraud)': f1_score(y_true, pred),
            'PR-AUC': pr_auc
        })

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(f"PR-AUC: {pr_auc:.4f}")
    print(f"\n--- Default threshold (0.5) ---")
    print(classification_report(y_true, pred_def, target_names=['Legit (0)', 'Fraud (1)']))
    print(f"--- F1-optimal threshold ({thr_arr[j1]:.4f}) ---")
    print(classification_report(y_true, pred_f1, target_names=['Legit (0)', 'Fraud (1)']))
    print(f"--- F2-optimal threshold ({thr_arr[j2]:.4f}) ---")
    print(classification_report(y_true, pred_f2, target_names=['Legit (0)', 'Fraud (1)']))

    return y_proba

print('evaluate() helper ready')

evaluate() helper ready


## 3. Focal Loss

Standard binary cross-entropy treats all samples equally.
Focal loss adds a modulating factor `(1 - p_t)^gamma` that
down-weights easy examples and focuses training on the hard
misclassified fraud cases — exactly what we need.

In [12]:
def focal_loss(gamma=2.0, alpha=0.25):
    """Focal loss for binary classification. alpha weights the positive class."""
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        focal_weight = alpha_t * tf.pow(1 - p_t, gamma)
        bce = -(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
        return tf.reduce_mean(focal_weight * bce)
    return loss_fn

print('Focal loss defined')

Focal loss defined


## 4. Experiment 1 — NN with Class Weights (multiple scales)

Test class weight ratios of 1x, 2x, and 3x the natural imbalance ratio
to see how aggressive weighting affects recall.

In [13]:
def build_nn(input_dim, hidden_sizes=[128, 64, 32], dropout=0.3):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for units in hidden_sizes:
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

base_ratio = neg_count / pos_count

for multiplier in [1, 2, 3]:
    weight = {0: 1.0, 1: base_ratio * multiplier}
    label = f'1. NN class_weight {multiplier}x ({weight[1]:.0f})'

    tf.random.set_seed(42)
    np.random.seed(42)
    model = build_nn(x_train.shape[1])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    model.fit(
        x_train, y_train,
        epochs=100, batch_size=2048,
        validation_data=(x_val, y_val),
        class_weight=weight,
        callbacks=[early_stop],
        verbose=0
    )
    proba = model.predict(x_val, verbose=0).ravel()
    evaluate(label, y_val, proba)


1. NN class_weight 1x (577)
PR-AUC: 0.7135

--- Default threshold (0.5) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.23      0.84      0.36        49

    accuracy                           0.99     28481
   macro avg       0.62      0.92      0.68     28481
weighted avg       1.00      0.99      1.00     28481

--- F1-optimal threshold (1.0000) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.88      0.76      0.81        49

    accuracy                           1.00     28481
   macro avg       0.94      0.88      0.91     28481
weighted avg       1.00      1.00      1.00     28481

--- F2-optimal threshold (1.0000) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.88      0.76      0.81        49

    accuracy                 

## 5. Experiment 2 — NN with Focal Loss

Focal loss with gamma=2 (hard example focus) and alpha=0.75 (heavy fraud weight).
Also test gamma=3 for even more aggressive hard-example mining.

In [14]:
for gamma, alpha in [(2.0, 0.75), (2.0, 0.9), (3.0, 0.75), (3.0, 0.9)]:
    label = f'2. Focal NN (g={gamma}, a={alpha})'

    tf.random.set_seed(42)
    np.random.seed(42)
    model = build_nn(x_train.shape[1])
    model.compile(optimizer='adam', loss=focal_loss(gamma=gamma, alpha=alpha), metrics=['AUC'])
    model.fit(
        x_train, y_train,
        epochs=100, batch_size=2048,
        validation_data=(x_val, y_val),
        callbacks=[early_stop],
        verbose=0
    )
    proba = model.predict(x_val, verbose=0).ravel()
    evaluate(label, y_val, proba)


2. Focal NN (g=2.0, a=0.75)
PR-AUC: 0.8223

--- Default threshold (0.5) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.88      0.78      0.83        49

    accuracy                           1.00     28481
   macro avg       0.94      0.89      0.91     28481
weighted avg       1.00      1.00      1.00     28481

--- F1-optimal threshold (0.5513) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.88      0.78      0.83        49

    accuracy                           1.00     28481
   macro avg       0.94      0.89      0.91     28481
weighted avg       1.00      1.00      1.00     28481

--- F2-optimal threshold (0.3147) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.77      0.84      0.80        49

    accuracy                 

## 6. Experiment 3 — SMOTE + NN (multiple ratios)

Oversample fraud to 30%, 50%, and 100% (fully balanced), then train.
No class weighting needed since SMOTE balances the data directly.

In [15]:
from imblearn.over_sampling import SMOTE

for ratio in [0.3, 0.5, 1.0]:
    smote = SMOTE(sampling_strategy=ratio, k_neighbors=5, random_state=42)
    x_sm, y_sm = smote.fit_resample(x_train, y_train)
    label = f'3. SMOTE {ratio} + NN'
    print(f"\n{label}: resampled shape {x_sm.shape}, fraud ratio {y_sm.mean():.2%}")

    tf.random.set_seed(42)
    np.random.seed(42)
    model = build_nn(x_train.shape[1])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    model.fit(
        x_sm, y_sm,
        epochs=100, batch_size=2048,
        validation_data=(x_val, y_val),
        callbacks=[early_stop],
        verbose=0
    )
    proba = model.predict(x_val, verbose=0).ravel()
    evaluate(label, y_val, proba)


3. SMOTE 0.3 + NN: resampled shape (295686, 30), fraud ratio 23.08%

3. SMOTE 0.3 + NN
PR-AUC: 0.7780

--- Default threshold (0.5) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      0.99      1.00     28432
   Fraud (1)       0.23      0.86      0.36        49

    accuracy                           0.99     28481
   macro avg       0.61      0.93      0.68     28481
weighted avg       1.00      0.99      1.00     28481

--- F1-optimal threshold (0.9957) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.88      0.78      0.83        49

    accuracy                           1.00     28481
   macro avg       0.94      0.89      0.91     28481
weighted avg       1.00      1.00      1.00     28481

--- F2-optimal threshold (0.9651) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.82  

## 7. Experiment 4 — SMOTE + Focal Loss NN

Combine SMOTE oversampling with focal loss — attack the problem
from both sides: more fraud examples AND harder gradient signal.

In [16]:
for ratio in [0.3, 0.5]:
    smote = SMOTE(sampling_strategy=ratio, k_neighbors=5, random_state=42)
    x_sm, y_sm = smote.fit_resample(x_train, y_train)

    for gamma, alpha in [(2.0, 0.75), (3.0, 0.9)]:
        label = f'4. SMOTE {ratio} + Focal(g={gamma},a={alpha})'

        tf.random.set_seed(42)
        np.random.seed(42)
        model = build_nn(x_train.shape[1])
        model.compile(optimizer='adam', loss=focal_loss(gamma=gamma, alpha=alpha), metrics=['AUC'])
        model.fit(
            x_sm, y_sm,
            epochs=100, batch_size=2048,
            validation_data=(x_val, y_val),
            callbacks=[early_stop],
            verbose=0
        )
        proba = model.predict(x_val, verbose=0).ravel()
        evaluate(label, y_val, proba)


4. SMOTE 0.3 + Focal(g=2.0,a=0.75)
PR-AUC: 0.7758

--- Default threshold (0.5) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      0.99      0.99     28432
   Fraud (1)       0.09      0.86      0.16        49

    accuracy                           0.98     28481
   macro avg       0.55      0.92      0.58     28481
weighted avg       1.00      0.98      0.99     28481

--- F1-optimal threshold (0.8984) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.87      0.80      0.83        49

    accuracy                           1.00     28481
   macro avg       0.93      0.90      0.91     28481
weighted avg       1.00      1.00      1.00     28481

--- F2-optimal threshold (0.8064) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.72      0.84      0.77        49

    accuracy          

## 8. Experiment 5 — Wider/Deeper Architectures

Test whether more model capacity helps catch the hard cases.

In [17]:
architectures = [
    ([256, 128, 64, 32], 0.3, 'Wide-deep'),
    ([64, 32, 16], 0.2, 'Narrow'),
    ([256, 128, 64, 32, 16], 0.4, 'Very deep'),
]

weight_2x = {0: 1.0, 1: base_ratio * 2}

for hidden, drop, arch_name in architectures:
    label = f'5. {arch_name} NN (2x weight)'

    tf.random.set_seed(42)
    np.random.seed(42)
    model = build_nn(x_train.shape[1], hidden_sizes=hidden, dropout=drop)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    model.fit(
        x_train, y_train,
        epochs=100, batch_size=2048,
        validation_data=(x_val, y_val),
        class_weight=weight_2x,
        callbacks=[early_stop],
        verbose=0
    )
    proba = model.predict(x_val, verbose=0).ravel()
    evaluate(label, y_val, proba)


5. Wide-deep NN (2x weight)
PR-AUC: 0.6801

--- Default threshold (0.5) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      0.99      0.99     28432
   Fraud (1)       0.13      0.86      0.22        49

    accuracy                           0.99     28481
   macro avg       0.56      0.92      0.61     28481
weighted avg       1.00      0.99      0.99     28481

--- F1-optimal threshold (0.8984) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.82      0.73      0.77        49

    accuracy                           1.00     28481
   macro avg       0.91      0.87      0.89     28481
weighted avg       1.00      1.00      1.00     28481

--- F2-optimal threshold (0.7311) ---
              precision    recall  f1-score   support

   Legit (0)       1.00      1.00      1.00     28432
   Fraud (1)       0.66      0.82      0.73        49

    accuracy                 

## 9. Experiment 6 — Ensemble: Best NN + RF + XGBoost

Different model families make different errors. Averaging their
probabilities can catch fraud cases that any single model misses.

In [18]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# RF
rf = RandomForestClassifier(
    n_estimators=300, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf.fit(x_train, y_train)
proba_rf = rf.predict_proba(x_val)[:, 1]
print(f"RF PR-AUC: {average_precision_score(y_val, proba_rf):.4f}")

# XGBoost
xgb = XGBClassifier(
    n_estimators=100, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg_count / pos_count,
    eval_metric='logloss', random_state=42
)
xgb.fit(x_train, y_train)
proba_xgb = xgb.predict_proba(x_val)[:, 1]
print(f"XGB PR-AUC: {average_precision_score(y_val, proba_xgb):.4f}")

# Find best NN from previous experiments
nn_results = [r for r in results if r['Threshold'].startswith('F2')]
best_nn_exp = max(nn_results, key=lambda r: r['Recall (fraud)'])['Experiment']
print(f"\nBest NN experiment: {best_nn_exp}")
print("Retraining best NN config for ensemble...")

# Retrain best NN: SMOTE 0.5 + focal loss as a strong default
smote_ens = SMOTE(sampling_strategy=0.5, k_neighbors=5, random_state=42)
x_sm_ens, y_sm_ens = smote_ens.fit_resample(x_train, y_train)

tf.random.set_seed(42)
np.random.seed(42)
nn_ens = build_nn(x_train.shape[1])
nn_ens.compile(optimizer='adam', loss=focal_loss(gamma=2.0, alpha=0.75), metrics=['AUC'])
nn_ens.fit(
    x_sm_ens, y_sm_ens,
    epochs=100, batch_size=2048,
    validation_data=(x_val, y_val),
    callbacks=[early_stop],
    verbose=0
)
proba_nn_ens = nn_ens.predict(x_val, verbose=0).ravel()
print(f"NN PR-AUC: {average_precision_score(y_val, proba_nn_ens):.4f}")

# Ensembles with different weight combos
combos = [
    (1/3, 1/3, 1/3, 'Equal'),
    (0.2, 0.3, 0.5, 'XGB-heavy'),
    (0.5, 0.25, 0.25, 'NN-heavy'),
    (0.4, 0.4, 0.2, 'NN+RF heavy'),
]
for w_nn, w_rf, w_xgb, combo_name in combos:
    proba_ens = w_nn * proba_nn_ens + w_rf * proba_rf + w_xgb * proba_xgb
    evaluate(f'6. Ensemble {combo_name}', y_val, proba_ens)

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/O

## 10. Results Summary

In [ ]:
results_df = pd.DataFrame(results)
for col in ['Precision (fraud)', 'Recall (fraud)', 'Recall (legit)', 'F1 (fraud)', 'PR-AUC']:
    results_df[col] = results_df[col].round(4)

print('=== F2-OPTIMAL RESULTS — ALL EXPERIMENTS (sorted by Fraud Recall) ===\n')
f2_df = results_df[results_df['Threshold'].str.startswith('F2')].sort_values(
    ['Recall (fraud)', 'F1 (fraud)'], ascending=[False, False]
)
print(f2_df.to_string(index=False))

best_row = f2_df.iloc[0]
print(f"\n*** BEST MODEL: {best_row['Experiment']}")
print(f"    Fraud Recall={best_row['Recall (fraud)']:.4f}  Legit Recall={best_row['Recall (legit)']:.4f}")
print(f"    Precision={best_row['Precision (fraud)']:.4f}  F1={best_row['F1 (fraud)']:.4f}  PR-AUC={best_row['PR-AUC']:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

f2_sorted = f2_df.sort_values('Recall (fraud)', ascending=True)

fig, ax = plt.subplots(figsize=(12, max(6, len(f2_sorted)*0.35)))
y_pos = range(len(f2_sorted))

colors = []
for exp in f2_sorted['Experiment']:
    if 'Ensemble' in exp:
        colors.append('#9b59b6')
    elif 'Focal' in exp:
        colors.append('#e74c3c')
    elif 'SMOTE' in exp:
        colors.append('#2ecc71')
    else:
        colors.append('#3498db')

bars = ax.barh(y_pos, f2_sorted['Recall (fraud)'], color=colors, edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(f2_sorted['Experiment'], fontsize=8)
ax.set_xlabel('Recall (Fraud Class)', fontsize=11)
ax.set_title('Fraud Recall — Neural Network Experiments (F2-Optimal Threshold)', fontsize=13)
ax.axvline(x=0.87, color='red', linestyle='--', linewidth=1)
ax.axvline(x=0.857, color='orange', linestyle=':', linewidth=1)

legend_handles = [
    Patch(facecolor='#3498db', edgecolor='black', label='Class-weighted NN'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='Focal loss NN'),
    Patch(facecolor='#2ecc71', edgecolor='black', label='SMOTE + NN'),
    Patch(facecolor='#9b59b6', edgecolor='black', label='Ensemble'),
    plt.Line2D([0], [0], color='red', linestyle='--', label='Target 0.87'),
    plt.Line2D([0], [0], color='orange', linestyle=':', label='RF ceiling 0.857'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=8)

for bar, val in zip(bars, f2_sorted['Recall (fraud)']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [19]:
# Extra: search threshold for recall ≥ 0.87 using focal NN (g=3.0, a=0.75)

tf.random.set_seed(42)
np.random.seed(42)

focal_best = build_nn(x_train.shape[1])
focal_best.compile(
    optimizer='adam',
    loss=focal_loss(gamma=3.0, alpha=0.75),
    metrics=['AUC'],
)

focal_best.fit(
    x_train, y_train,
    epochs=100,
    batch_size=2048,
    validation_data=(x_val, y_val),
    callbacks=[early_stop],
    verbose=0,
)

proba_best = focal_best.predict(x_val, verbose=0).ravel()

from sklearn.metrics import precision_recall_curve, classification_report

prec, rec, thr = precision_recall_curve(y_val, proba_best)

best_thr = None
best_p = None
best_r = None

for p, r, t in zip(prec[1:], rec[1:], thr):
    if r >= 0.87:
        best_thr = t
        best_p = p
        best_r = r
        break

if best_thr is not None:
    y_pred = (proba_best >= best_thr).astype(int)
    print(f"Threshold achieving recall ≥ 0.87: {best_thr:.4f}")
    print(f"Precision={best_p:.4f}, Recall={best_r:.4f}")
    print(classification_report(y_val, y_pred, target_names=['Legit (0)', 'Fraud (1)']))
else:
    print("No threshold on this focal NN achieves recall ≥ 0.87 on this validation split.")

Threshold achieving recall ≥ 0.87: 0.0000
Precision=0.0017, Recall=1.0000
              precision    recall  f1-score   support

   Legit (0)       0.00      0.00      0.00     28432
   Fraud (1)       0.00      1.00      0.00        49

    accuracy                           0.00     28481
   macro avg       0.00      0.50      0.00     28481
weighted avg       0.00      0.00      0.00     28481

